# 5. The complete tool-calling agent

**LangGraph tutorial, lesson 5 of 5**

Everything assembles here: the tools from lesson 2, the loop from lesson 3, and
the state/node/edge vocabulary from lesson 4.

The shape below is the classic two-node agent. Memorise it — almost every
tool-calling agent you will read is a variation on it.

```
       agent  ──tool calls?──>  tools
         ^                        │
         └────── results ─────────┘
         │
         └──no tool calls──> END
```

The cycle between `agent` and `tools` **is** the `while` loop from lesson 3,
drawn as an edge. That is the whole trick.

> **On the word "DAG":** this graph is deliberately **not** acyclic — the loop is
> the point, and allowing cycles is exactly what separates LangGraph from a plain
> DAG pipeline. What bounds the looping is `recursion_limit`, not the shape.

In [ ]:
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

from common import make_llm, text_of
from tools import ALL_TOOLS

SYSTEM_PROMPT = (
    "You are a concise assistant with access to tools. "
    "Always use a tool rather than guessing: you have no clock of your own, no "
    "access to employee records, and you must never do arithmetic in your head. "
    "When you have everything you need, answer in plain prose."
)

## State

Just a message list. Most agents need nothing more. Anything extra you want to
thread through — a user id, a scratchpad, a budget counter — becomes another key.

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

## Node 1: `agent` — ask the model what to do next

One model call. Returns either a final answer or tool requests.

In [ ]:
llm_with_tools = make_llm().bind_tools(ALL_TOOLS)


def agent_node(state: AgentState) -> dict:
    reply = llm_with_tools.invoke([SystemMessage(SYSTEM_PROMPT), *state["messages"]])
    return {"messages": [reply]}

## Node 2: `tools` — run whatever the model asked for

LangGraph ships a prebuilt `ToolNode` that does exactly this, and in real code
you would use it:

```python
from langgraph.prebuilt import ToolNode
builder.add_node("tools", ToolNode(ALL_TOOLS))
```

We write it longhand **once** so there is no mystery about what it does. It is
the body of lesson 3's inner `for` loop, unchanged.

In [ ]:
tools_by_name = {t.name: t for t in ALL_TOOLS}


def tool_node(state: AgentState) -> dict:
    """Execute every tool call on the most recent message."""
    last_message = state["messages"][-1]
    results = []

    for request in last_message.tool_calls:
        tool = tools_by_name.get(request["name"])
        if tool is None:
            output = f"Error: no such tool {request['name']!r}."
        else:
            try:
                output = tool.invoke(request["args"])
            except Exception as exc:
                # Never let a tool exception kill the graph. Hand the error back
                # as text and the model will usually correct itself.
                output = f"Error running {request['name']}: {exc}"

        print(f"    [tool] {request['name']}({request['args']})")
        results.append(ToolMessage(content=str(output), tool_call_id=request["id"]))

    return {"messages": results}

## The conditional edge — the one real decision

A normal edge says "always go here next". A **conditional edge** runs a function
that inspects state and returns the name of the next node.

Note it is *not* a node: it adds nothing to state and makes no model call. It is
pure routing, and must be fast and side-effect free.

In [ ]:
def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    """Tool calls pending? Go run them. Otherwise we are finished."""
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "tools"
    return END


# `tools_condition` from langgraph.prebuilt is this exact function, if you would
# rather not write it yourself.

## Wire it up

Two things to notice:

- `add_conditional_edges` takes a dict mapping the router's return value to a
  destination node.
- The edge from `tools` back to `agent` is what **closes the loop**.

In [ ]:
builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")

builder.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", END: END},
)

# After running tools, ALWAYS go back to the model so it can read the results.
builder.add_edge("tools", "agent")

agent = builder.compile()
print(agent.get_graph().draw_ascii())

## Run it

The question below is chosen so that **all three tools are genuinely required**,
and each depends on the one before it:

| Tool | Why it is unavoidable |
|---|---|
| `lookup_employee` | Her start year is not in the model's knowledge |
| `get_current_time` | The model has no clock |
| `calculator` | The multiplication by 365.25 |

We use `.stream(stream_mode="updates")` to yield one item per node execution, so
you can watch the `agent`/`tools` cycle turn. `.invoke()` would only give the
final state — fine in production, useless for learning.

In [ ]:
QUESTION = (
    "What is Ada Lovelace's tenure in whole years as of today, and what is that "
    "tenure expressed in days, assuming 365.25 days per year?"
)

step = 0
final_message = None

for update in agent.stream(
    {"messages": [HumanMessage(QUESTION)]},
    # Safety bound: counts every node visit, so a 3-tool answer uses ~6.
    # Without it, a confused model could loop forever.
    {"recursion_limit": 25},
    stream_mode="updates",
):
    for node_name, node_output in update.items():
        step += 1
        message = node_output["messages"][-1]
        if node_name == "agent":
            requested = getattr(message, "tool_calls", [])
            if requested:
                print(f"  {step}. [agent] wants: {', '.join(r['name'] for r in requested)}")
            else:
                print(f"  {step}. [agent] final answer ready")
        else:
            print(f"  {step}. [tools] returned {len(node_output['messages'])} result(s)")
        final_message = message

print("\n" + "=" * 60)
print(text_of(final_message))

## Read that trace again

**The loop turned twice**, and that is the part worth studying.

The model could not call `calculator` on the first pass — it did not yet know the
start year or the current year. So it fired the two independent lookups
together, waited for both, *then* computed.

**Nobody wrote that plan.** It falls out of the cycle.

## Try it yourself

Change the question and re-run. A simpler one takes a shorter path through the
very same graph.

In [ ]:
def ask(question: str) -> str:
    """Convenience wrapper: run the agent, return the final text."""
    result = agent.invoke(
        {"messages": [HumanMessage(question)]},
        {"recursion_limit": 25},
    )
    return text_of(result["messages"][-1])


print(ask("What time is it in Tokyo right now?"))

## Where to go next

- **Swap in the prebuilts.** Replace `tool_node` with `ToolNode(ALL_TOOLS)` and
  `should_continue` with `tools_condition`. Identical behaviour, less code — but
  now you know what they do.
- **Add memory across invocations** with a checkpointer:
  ```python
  from langgraph.checkpoint.memory import InMemorySaver
  agent = builder.compile(checkpointer=InMemorySaver())
  agent.invoke(..., {"configurable": {"thread_id": "abc"}})
  ```
- **Add human-in-the-loop** with `builder.compile(interrupt_before=["tools"])`,
  so a person approves each tool call. This is the feature that most justifies
  the graph in the first place.
- **Write a fourth tool** in `tools.py`, add it to `ALL_TOOLS`, and re-run with
  no other changes. Watch the agent pick it up on its own — that single
  experiment is the argument for this whole architecture.